<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/eshan-dev/notebooks/04_question_intent_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Question Answering Assistant

## Notebook 04 – Question Intent Dataset Creation

### Objective

This notebook creates a supervised question-intent dataset for training the Random Forest and LSTM intent-classification models.

The notebook will:

1. Load the cleaned football datasets.
2. Define the supported question intents.
3. Create multiple natural-language question templates for each intent.
4. Populate the templates using real football data.
5. Create a dataset containing questions and their corresponding intent labels.
6. inspect the class distribution and data quality.
7. Split the dataset into training, validation, and testing sets.
8. Save the generated dataset and supporting artifacts.

In [1]:
# Clone the GitHub repository using the working branch
!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git

Cloning into 'football-qa-nlp'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 160 (delta 23), reused 29 (delta 14), pack-reused 112 (from 4)
Receiving objects: 100% (160/160), 28.99 MiB | 8.51 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
# Move the Colab working directory into the repository
%cd /content/football-qa-nlp

/content/football-qa-nlp


In [3]:
# Display the main folders and files in the repository
!ls

data  docs  models  notebooks  README.md  requirements.txt  src


In [4]:
# Display all files inside the data folder
!ls data

former_names_clean.csv	goalscorers.csv    shootouts_clean.csv
former_names.csv	results_clean.csv  shootouts.csv
goalscorers_clean.csv	results.csv


In [5]:
# Display the saved feature-engineering artifacts
!ls models/features

football_corpus_clean.csv  lstm_tokenizer.pkl	    tfidf_features.npz
lstm_padded_sequences.npy  max_sequence_length.pkl  tfidf_vectorizer.pkl


In [6]:
# Import the libraries required for dataset creation and analysis
import os
import random
import numpy as np
import pandas as pd

# Display all columns clearly when inspecting DataFrames
pd.set_option("display.max_columns", None)

# Set a fixed random seed for reproducible question generation
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [7]:
# Define the main project paths
project_path = "/content/football-qa-nlp"
data_path = os.path.join(project_path, "data")
features_path = os.path.join(project_path, "models", "features")

# Define paths to the cleaned datasets
results_file = os.path.join(data_path, "results_clean.csv")
scorers_file = os.path.join(data_path, "goalscorers_clean.csv")
shootouts_file = os.path.join(data_path, "shootouts_clean.csv")
former_names_file = os.path.join(data_path, "former_names_clean.csv")

In [8]:
# Store all required files in a dictionary for validation
required_files = {
    "Results dataset": results_file,
    "Goalscorers dataset": scorers_file,
    "Shootouts dataset": shootouts_file,
    "Former names dataset": former_names_file
}

# Check whether every required file exists
for file_name, file_path in required_files.items():
    if os.path.exists(file_path):
        print(f"Found: {file_name}")
    else:
        print(f"Missing: {file_name} -> {file_path}")

Found: Results dataset
Found: Goalscorers dataset
Found: Shootouts dataset
Found: Former names dataset


In [9]:
!ls data

former_names_clean.csv	goalscorers.csv    shootouts_clean.csv
former_names.csv	results_clean.csv  shootouts.csv
goalscorers_clean.csv	results.csv


In [10]:
!ls models/features

football_corpus_clean.csv  lstm_tokenizer.pkl	    tfidf_features.npz
lstm_padded_sequences.npy  max_sequence_length.pkl  tfidf_vectorizer.pkl


In [11]:
# Load the cleaned football datasets
results_clean = pd.read_csv(
    results_file,
    parse_dates=["date"]
)

scorers_clean = pd.read_csv(
    scorers_file,
    parse_dates=["date"]
)

shootouts_clean = pd.read_csv(
    shootouts_file,
    parse_dates=["date"]
)

former_names_clean = pd.read_csv(former_names_file)

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


In [12]:
# Store the loaded datasets in a dictionary for convenient inspection
datasets = {
    "Match results": results_clean,
    "Goal scorers": scorers_clean,
    "Penalty shootouts": shootouts_clean,
    "Former country names": former_names_clean
}

# Display the shape and columns of each dataset
for dataset_name, dataframe in datasets.items():
    print(f"\n{dataset_name}")
    print("-" * len(dataset_name))
    print(f"Shape: {dataframe.shape}")
    print(f"Columns: {dataframe.columns.tolist()}")


Match results
-------------
Shape: (49485, 9)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Goal scorers
------------
Shape: (47855, 8)
Columns: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']

Penalty shootouts
-----------------
Shape: (682, 5)
Columns: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']

Former country names
--------------------
Shape: (36, 4)
Columns: ['current', 'former', 'start_date', 'end_date']


In [13]:
# Display two sample records from each cleaned dataset
for dataset_name, dataframe in datasets.items():
    print(f"\n{dataset_name}")
    display(dataframe.head(2))


Match results


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False



Goal scorers


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55,False,False



Penalty shootouts


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,Unknown
1,1971-11-14,South Korea,Vietnam Republic,South Korea,Unknown



Former country names


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04


## 2. Supported Question Intent Catalogue

The Football Question Answering Assistant supports multiple question intents based on match results, goalscorer records, and penalty shootout data.

Each intent represents the specific type of information requested by the user. The intent classifier will predict one of these labels before the application retrieves the appropriate answer from the structured football datasets.

In [14]:
# Define the supported question intents and their metadata
intent_catalogue = {
    "match_score": {
        "dataset": "results",
        "description": "Requests the final score of a specific match."
    },
    "match_winner": {
        "dataset": "results",
        "description": "Requests the winner of a specific match or whether it ended in a draw."
    },
    "home_team_score": {
        "dataset": "results",
        "description": "Requests the number of goals scored by the home team."
    },
    "away_team_score": {
        "dataset": "results",
        "description": "Requests the number of goals scored by the away team."
    },
    "total_goals": {
        "dataset": "results",
        "description": "Requests the total number of goals scored in a match."
    },
    "match_date": {
        "dataset": "results",
        "description": "Requests the date on which a match was played."
    },
    "match_location": {
        "dataset": "results",
        "description": "Requests the city and country where a match was played."
    },
    "tournament": {
        "dataset": "results",
        "description": "Requests the tournament or competition of a match."
    },
    "neutral_status": {
        "dataset": "results",
        "description": "Requests whether a match was played at a neutral venue."
    },
    "head_to_head_matches": {
        "dataset": "results",
        "description": "Requests historical matches played between two teams."
    },
    "team_match_history": {
        "dataset": "results",
        "description": "Requests historical or recent matches involving one team."
    },
    "team_wins": {
        "dataset": "results",
        "description": "Requests the number of matches won by a team."
    },
    "team_goals_scored": {
        "dataset": "results",
        "description": "Requests the total number of goals scored by a team."
    },
    "team_goals_conceded": {
        "dataset": "results",
        "description": "Requests the total number of goals conceded by a team."
    },
    "scorer": {
        "dataset": "goalscorers",
        "description": "Requests the player or players who scored in a match."
    },
    "goal_minute": {
        "dataset": "goalscorers",
        "description": "Requests the minute in which a player scored."
    },
    "penalty_status": {
        "dataset": "goalscorers",
        "description": "Requests whether a goal was scored from a penalty."
    },
    "own_goal_status": {
        "dataset": "goalscorers",
        "description": "Requests whether a goal was recorded as an own goal."
    },
    "player_goal_count": {
        "dataset": "goalscorers",
        "description": "Requests the number of goals scored by a player."
    },
    "shootout_winner": {
        "dataset": "shootouts",
        "description": "Requests the winner of a penalty shootout."
    },
    "first_shooter": {
        "dataset": "shootouts",
        "description": "Requests the team that took the first kick in a penalty shootout."
    }
}

print(f"Total supported intents: {len(intent_catalogue)}")

Total supported intents: 21


In [15]:
# Convert the intent catalogue into a DataFrame for easier inspection
intent_catalogue_df = (
    pd.DataFrame.from_dict(intent_catalogue, orient="index")
    .reset_index()
    .rename(columns={"index": "intent"})
)

display(intent_catalogue_df)

,intent,dataset,description
0,match_score,results,Requests the final score of a specific match.
1,match_winner,results,Requests the winner of a specific match or whe...
2,home_team_score,results,Requests the number of goals scored by the hom...
3,away_team_score,results,Requests the number of goals scored by the awa...
4,total_goals,results,Requests the total number of goals scored in a...
5,match_date,results,Requests the date on which a match was played.
6,match_location,results,Requests the city and country where a match wa...
7,tournament,results,Requests the tournament or competition of a ma...
8,neutral_status,results,Requests whether a match was played at a neutr...
9,head_to_head_matches,results,Requests historical matches played between two...


In [16]:
# Count how many intents are associated with each source dataset
intent_source_distribution = (
    intent_catalogue_df["dataset"]
    .value_counts()
    .rename_axis("dataset")
    .reset_index(name="number_of_intents")
)

display(intent_source_distribution)

,dataset,number_of_intents
0,results,14
1,goalscorers,5
2,shootouts,2


In [17]:
# Recreate the intent catalogue DataFrame cleanly
intent_catalogue_df = pd.DataFrame(
    [
        {
            "intent": intent_name,
            "dataset": metadata["dataset"],
            "description": metadata["description"]
        }
        for intent_name, metadata in intent_catalogue.items()
    ]
)

# Display the cleaned catalogue
display(intent_catalogue_df)

# Verify the final column names
print("Columns:", intent_catalogue_df.columns.tolist())
print("Shape:", intent_catalogue_df.shape)

,intent,dataset,description
0,match_score,results,Requests the final score of a specific match.
1,match_winner,results,Requests the winner of a specific match or whe...
2,home_team_score,results,Requests the number of goals scored by the hom...
3,away_team_score,results,Requests the number of goals scored by the awa...
4,total_goals,results,Requests the total number of goals scored in a...
5,match_date,results,Requests the date on which a match was played.
6,match_location,results,Requests the city and country where a match wa...
7,tournament,results,Requests the tournament or competition of a ma...
8,neutral_status,results,Requests whether a match was played at a neutr...
9,head_to_head_matches,results,Requests historical matches played between two...


Columns: ['intent', 'dataset', 'description']
Shape: (21, 3)


## 3. Question Template Design

Multiple natural-language templates are defined for each supported intent.

The templates represent different ways users may request the same type of football information. Real teams, players, dates, tournaments, and match details from the cleaned datasets will later replace the placeholders.

Using multiple templates reduces dependence on one fixed sentence structure and helps the intent-classification models generalize to unseen user questions.

In [25]:
# Store all question templates for each supported intent
question_templates = {}

In [29]:
# Question templates for the match_score intent
question_templates["match_score"] = [

    "What was the score between {home_team} and {away_team}?",

    "What was the final score between {home_team} and {away_team}?",

    "How did {home_team} vs {away_team} finish?",

    "What was the result of {home_team} against {away_team}?",

    "Can you tell me the score between {home_team} and {away_team}?",

    "What was the scoreline for {home_team} versus {away_team}?",

    "What was the final score when {home_team} played {away_team}?",

    "How did the match between {home_team} and {away_team} end?",

    "What was the final result when {home_team} played {away_team}?",

    "What score did {home_team} and {away_team} finish with?",

    "What was the score between {home_team} and {away_team} in {tournament}?",

    "What was the score between {home_team} and {away_team} on {date}?"
]

In [27]:
# Question templates for the match_winner intent
question_templates["match_winner"] = [

    "Who won the match between {home_team} and {away_team}?",

    "Which team won between {home_team} and {away_team}?",

    "Who was victorious in the match between {home_team} and {away_team}?",

    "Did {home_team} or {away_team} win the match?",

    "Who came out on top between {home_team} and {away_team}?",

    "Which team defeated the other in {home_team} versus {away_team}?",

    "Who was the winner when {home_team} played {away_team}?",

    "Who won the {tournament} match between {home_team} and {away_team}?",

    "Who won the match between {home_team} and {away_team} on {date}?",

    "Which side won when {home_team} faced {away_team}?",

    "Who emerged as the winner in {home_team} vs {away_team}?",

    "Did the match between {home_team} and {away_team} end in a draw or did someone win?"
]

In [28]:
print("Available intents:")
print(question_templates.keys())

print()

print("Match score templates:", len(question_templates["match_score"]))
print("Match winner templates:", len(question_templates["match_winner"]))

Available intents:
dict_keys(['match_score', 'match_winner'])

Match score templates: 12
Match winner templates: 12


## 4. Reusable Question Generation Functions

Reusable functions are created to transform structured football records into natural-language questions.

Each function selects a question template, replaces its placeholders with real dataset values, and returns a labeled question that can be used for supervised intent classification.

In [30]:
# Generate one natural-language question from a match record
def generate_match_question(row, intent):
    """
    Generate one question for a match-related intent.

    Parameters
    ----------
    row : pandas.Series
        One record from the cleaned match-results dataset.

    intent : str
        The intent label whose question template should be used.

    Returns
    -------
    str
        A completed natural-language football question.
    """

    # Check whether templates exist for the requested intent
    if intent not in question_templates:
        raise ValueError(f"No question templates found for intent: {intent}")

    # Randomly select one template belonging to the requested intent
    template = random.choice(question_templates[intent])

    # Convert the match date into a readable YYYY-MM-DD string
    formatted_date = row["date"].strftime("%Y-%m-%d")

    # Replace the placeholders with real values from the match record
    question = template.format(
        home_team=row["home_team"],
        away_team=row["away_team"],
        tournament=row["tournament"],
        date=formatted_date
    )

    return question

In [31]:
# Select one real match record for testing
sample_match = results_clean.iloc[0]

# Generate one example for each currently available match intent
sample_score_question = generate_match_question(
    sample_match,
    "match_score"
)

sample_winner_question = generate_match_question(
    sample_match,
    "match_winner"
)

print("Match record:")
display(sample_match.to_frame().T)

print("\nGenerated match-score question:")
print(sample_score_question)

print("\nGenerated match-winner question:")
print(sample_winner_question)

Match record:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30 00:00:00,Scotland,England,0,0,Friendly,Glasgow,Scotland,False



Generated match-score question:
What was the score between Scotland and England in Friendly?

Generated match-winner question:
Which team won between Scotland and England?


In [125]:
# Question templates for the home_team_score intent
question_templates["home_team_score"] = [
    "How many goals did the home team {home_team} score against {away_team}?",
    "What was the home team's score when {home_team} played {away_team}?",
    "How many goals did the home team {home_team} score in this match?",
    "What was the home score when {home_team} played {away_team}?",
    "How many times did the home team {home_team} score against {away_team}?",
    "What score did the home team {home_team} record against {away_team}?",
    "How many goals were scored by the home team {home_team} against {away_team}?",
    "What was the home team's goal total against {away_team}?",
    "How many goals did the home team {home_team} score in the {tournament} match against {away_team}?",
    "What was the home team's score in {home_team} versus {away_team}?",
    "How many goals did the home team {home_team} score while hosting {away_team}?",
    "What was the home team's score when {home_team} played {away_team} on {date}?"
]

In [126]:
# Question templates for the away_team_score intent
question_templates["away_team_score"] = [
    "What was the away team's score when {home_team} played {away_team}?",
    "What was the away team {away_team}'s score against {home_team}?",
    "How many goals did the away team {away_team} score in this match?",
    "What was the away score when {home_team} played {away_team}?",
    "How many times did the away team {away_team} score against {home_team}?",
    "What score did the away team {away_team} record against {home_team}?",
    "How many goals were scored by the away team {away_team} against {home_team}?",
    "What was the away team's goal total against {home_team}?",
    "How many goals did the away team {away_team} score in the {tournament} match against {home_team}?",
    "What was the away team's score in {home_team} versus {away_team}?",
    "How many goals did the away team {away_team} score while playing away to {home_team}?",
    "What was the away team's score when {home_team} played {away_team} on {date}?"
]

In [37]:
# Question templates for the total_goals intent
question_templates["total_goals"] = [
    "How many total goals were scored between {home_team} and {away_team}?",
    "How many goals were there in the match between {home_team} and {away_team}?",
    "What was the total number of goals in {home_team} versus {away_team}?",
    "How many goals were scored altogether when {home_team} played {away_team}?",
    "What was the total number of goals scored between {home_team} and {away_team}?",
    "How many goals did both teams score in total?",
    "What was the total goal count in the match between {home_team} and {away_team}?",
    "How many goals were recorded in {home_team} vs {away_team}?",
    "What was the combined number of goals in the {tournament} match between {home_team} and {away_team}?",
    "How many goals were scored in total when {home_team} faced {away_team}?",
    "What was the overall goal total for the match?",
    "How many total goals were scored between {home_team} and {away_team} on {date}?"
]

In [48]:
# List of match-specific intents currently supported
current_intents = [
    "match_score",
    "match_winner",
    "home_team_score",
    "away_team_score",
    "total_goals",
    "match_date",
    "match_location",
    "tournament",
    "neutral_status"
]

for intent in current_intents:
    print(f"{intent}: {len(question_templates[intent])} templates")

match_score: 12 templates
match_winner: 12 templates
home_team_score: 12 templates
away_team_score: 12 templates
total_goals: 12 templates
match_date: 8 templates
match_location: 8 templates
tournament: 8 templates
neutral_status: 8 templates


In [36]:
# Generate sample questions for the three newly added intents
for intent in [
    "home_team_score",
    "away_team_score",
    "total_goals"
]:
    generated_question = generate_match_question(
        sample_match,
        intent
    )

    print(f"{intent}:")
    print(generated_question)
    print()

home_team_score:
How many goals did Scotland score against England?

away_team_score:
What was England's score against Scotland on 1872-11-30?

total_goals:
What was the combined goal total for Scotland and England?



## 5. Answer Generation Functions

In addition to generating natural-language questions, reusable functions are created to generate the corresponding expected answers.

These answers will later be used to validate the Question Answering Assistant and create a complete labelled question-answer dataset.

In [47]:
# Generate the expected answer for a match-related intent
def generate_match_answer(row, intent):

    if intent == "match_score":
        return (
            f"{row['home_team']} {row['home_score']} - "
            f"{row['away_score']} {row['away_team']}"
        )

    elif intent == "match_winner":
        if row["home_score"] > row["away_score"]:
            return row["home_team"]

        elif row["away_score"] > row["home_score"]:
            return row["away_team"]

        else:
            return "Draw"

    elif intent == "home_team_score":
        return str(row["home_score"])

    elif intent == "away_team_score":
        return str(row["away_score"])

    elif intent == "total_goals":
        return str(row["home_score"] + row["away_score"])

    elif intent == "match_date":
        return row["date"].strftime("%Y-%m-%d")

    elif intent == "match_location":
        return f"{row['city']}, {row['country']}"

    elif intent == "tournament":
        return str(row["tournament"])

    elif intent == "neutral_status":
        return "Yes" if row["neutral"] else "No"

    else:
        raise ValueError(
            f"Unsupported match-related intent: {intent}"
        )

In [39]:
# Test answer generation for the current intents
for intent in current_intents:

    answer = generate_match_answer(
        sample_match,
        intent
    )

    print(intent)
    print(answer)
    print()

match_score
Scotland 0 - 0 England

match_winner
Draw

home_team_score
0

away_team_score
0

total_goals
0



In [40]:
# Create one complete question-answer record for a match-related intent
def create_match_qa_record(row, intent):
    """
    Create one labelled question-answer record from a match result.

    Parameters
    ----------
    row : pandas.Series
        One row from the cleaned match-results dataset.

    intent : str
        The intent label used to generate the question and answer.

    Returns
    -------
    dict
        A dictionary containing the question, intent, answer,
        and source dataset.
    """

    question = generate_match_question(row, intent)
    answer = generate_match_answer(row, intent)

    qa_record = {
        "question": question,
        "intent": intent,
        "answer": answer,
        "source_dataset": "results"
    }

    return qa_record

In [41]:
# Create one complete example record
sample_qa_record = create_match_qa_record(
    sample_match,
    "match_winner"
)

print(sample_qa_record)

{'question': 'Did Scotland or England win the match?', 'intent': 'match_winner', 'answer': 'Draw', 'source_dataset': 'results'}


In [42]:
# Generate one complete record for every currently supported match intent
sample_qa_records = []

for intent in current_intents:
    record = create_match_qa_record(
        sample_match,
        intent
    )

    sample_qa_records.append(record)

# Convert the generated records into a DataFrame
sample_qa_df = pd.DataFrame(sample_qa_records)

display(sample_qa_df)

,question,intent,answer,source_dataset
0,What was the result of Scotland against England?,match_score,Scotland 0 - 0 England,results
1,Who was victorious in the match between Scotla...,match_winner,Draw,results
2,What was Scotland's score against England on 1...,home_team_score,0,results
3,What was England's score against Scotland?,away_team_score,0,results
4,What was the overall goal total for the match?,total_goals,0,results


In [43]:
# Question templates for the match_date intent
question_templates["match_date"] = [
    "When was the match between {home_team} and {away_team} played?",
    "What was the date of {home_team} versus {away_team}?",
    "On what date did {home_team} play {away_team}?",
    "When did {home_team} face {away_team}?",
    "What day was the match between {home_team} and {away_team}?",
    "When was the {tournament} match between {home_team} and {away_team} held?",
    "What date did {home_team} and {away_team} play each other?",
    "When did the match between {home_team} and {away_team} take place?"
]

In [44]:
# Question templates for the match_location intent
question_templates["match_location"] = [
    "Where was the match between {home_team} and {away_team} played?",
    "What was the location of {home_team} versus {away_team}?",
    "In which city and country did {home_team} play {away_team}?",
    "Where did {home_team} face {away_team}?",
    "Which location hosted the match between {home_team} and {away_team}?",
    "Where was the {tournament} match between {home_team} and {away_team} held?",
    "In what city was {home_team} against {away_team} played?",
    "What city and country hosted {home_team} versus {away_team}?"
]

In [45]:
# Question templates for the tournament intent
question_templates["tournament"] = [
    "Which tournament was the match between {home_team} and {away_team} part of?",
    "What competition did {home_team} and {away_team} play in?",
    "Which tournament featured {home_team} versus {away_team}?",
    "What competition was the match between {home_team} and {away_team} played in?",
    "In which tournament did {home_team} face {away_team}?",
    "What was the competition for {home_team} against {away_team} on {date}?",
    "Which event included the match between {home_team} and {away_team}?",
    "Which tournament included the match between {home_team} and {away_team}?"
]

In [46]:
# Question templates for the neutral_status intent
question_templates["neutral_status"] = [
    "Was the match between {home_team} and {away_team} played at a neutral venue?",
    "Did {home_team} and {away_team} play on neutral ground?",
    "Was {home_team} versus {away_team} held at a neutral location?",
    "Was the match between {home_team} and {away_team} played on neutral soil?",
    "Did either {home_team} or {away_team} have home advantage?",
    "Was the {tournament} match between {home_team} and {away_team} neutral?",
    "Was {home_team} against {away_team} played away from both teams' home grounds?",
    "Was the venue neutral for the match between {home_team} and {away_team}?"
]

In [49]:
# Generate one sample record for every current match intent
sample_qa_records = []

for intent in current_intents:
    record = create_match_qa_record(
        sample_match,
        intent
    )

    sample_qa_records.append(record)

sample_qa_df = pd.DataFrame(sample_qa_records)

display(sample_qa_df)

,question,intent,answer,source_dataset
0,What was the score between Scotland and Englan...,match_score,Scotland 0 - 0 England,results
1,Who won the match between Scotland and England...,match_winner,Draw,results
2,What was Scotland's score against England?,home_team_score,0,results
3,What was the away team's score in Scotland ver...,away_team_score,0,results
4,What was the total goal count in the match bet...,total_goals,0,results
5,When was the match between Scotland and Englan...,match_date,1872-11-30,results
6,Where was the match between Scotland and Engla...,match_location,"Glasgow, Scotland",results
7,What competition did Scotland and England play...,tournament,Friendly,results
8,Was the match between Scotland and England pla...,neutral_status,No,results


In [50]:
# Question templates for the head_to_head_matches intent
question_templates["head_to_head_matches"] = [
    "Show the matches played between {team_1} and {team_2}.",
    "What matches have {team_1} and {team_2} played against each other?",
    "Give me the head-to-head history between {team_1} and {team_2}.",
    "How many times have {team_1} and {team_2} played each other?",
    "List the previous meetings between {team_1} and {team_2}.",
    "What is the match history between {team_1} and {team_2}?",
    "Show the past results between {team_1} and {team_2}.",
    "How often have {team_1} and {team_2} faced each other?"
]

In [51]:
# Question templates for the team_match_history intent
question_templates["team_match_history"] = [
    "Show the recent matches played by {team}.",
    "What are {team}'s latest match results?",
    "List the most recent matches involving {team}.",
    "Show {team}'s match history.",
    "What matches has {team} played recently?",
    "Give me the latest results for {team}.",
    "What were {team}'s recent football matches?",
    "Show the last few matches played by {team}."
]

In [52]:
# Question templates for the team_wins intent
question_templates["team_wins"] = [
    "How many matches did {team} win in {year}?",
    "How many victories did {team} record in {year}?",
    "What was {team}'s number of wins in {year}?",
    "How many games did {team} win during {year}?",
    "How many matches did {team} win in {tournament}?",
    "What was {team}'s win total in {tournament}?",
    "How many victories did {team} achieve in {tournament}?",
    "How many wins did {team} have in the {tournament}?"
]

In [53]:
# Question templates for the team_goals_scored intent
question_templates["team_goals_scored"] = [
    "How many goals did {team} score in {year}?",
    "What was {team}'s total number of goals in {year}?",
    "How many goals did {team} score during {year}?",
    "What was {team}'s goal total in {year}?",
    "How many goals did {team} score in {tournament}?",
    "What was {team}'s total goal count in {tournament}?",
    "How many times did {team} score in the {tournament}?",
    "What was the number of goals scored by {team} in {tournament}?"
]

In [54]:
# Question templates for the team_goals_conceded intent
question_templates["team_goals_conceded"] = [
    "How many goals did {team} concede in {year}?",
    "What was {team}'s total number of goals conceded in {year}?",
    "How many goals did opponents score against {team} in {year}?",
    "What was {team}'s goals-conceded total in {year}?",
    "How many goals did {team} concede in {tournament}?",
    "What was {team}'s total number of goals allowed in {tournament}?",
    "How many goals were scored against {team} in the {tournament}?",
    "What was the number of goals conceded by {team} in {tournament}?"
]

In [55]:
# Verify the aggregation and history template counts
history_intents = [
    "head_to_head_matches",
    "team_match_history",
    "team_wins",
    "team_goals_scored",
    "team_goals_conceded"
]

for intent in history_intents:
    print(f"{intent}: {len(question_templates[intent])} templates")

head_to_head_matches: 8 templates
team_match_history: 8 templates
team_wins: 8 templates
team_goals_scored: 8 templates
team_goals_conceded: 8 templates


In [60]:
# Generate one question and record which filter type it uses
def generate_aggregation_question(
    intent,
    team=None,
    team_1=None,
    team_2=None,
    year=None,
    tournament=None
):
    """
    Generate one history or aggregation question and identify
    the filter used by the selected template.

    Returns
    -------
    tuple
        A completed question and its selected filter type.
    """

    # Confirm that templates exist for the requested intent
    if intent not in question_templates:
        raise ValueError(
            f"No question templates found for intent: {intent}"
        )

    # Randomly select one template for the requested intent
    template = random.choice(question_templates[intent])

    # Identify which filtering placeholder is used
    if "{year}" in template:
        filter_type = "year"

    elif "{tournament}" in template:
        filter_type = "tournament"

    else:
        filter_type = "none"

    # Replace the template placeholders with real values
    question = template.format(
        team=team,
        team_1=team_1,
        team_2=team_2,
        year=year,
        tournament=tournament
    )

    return question, filter_type

In [57]:
# Generate one sample head-to-head question
sample_head_to_head_question = generate_aggregation_question(
    intent="head_to_head_matches",
    team_1="Scotland",
    team_2="England"
)

print(sample_head_to_head_question)

How many times have Scotland and England played each other?


In [58]:
# Generate one sample team-match-history question
sample_team_history_question = generate_aggregation_question(
    intent="team_match_history",
    team="Scotland"
)

print(sample_team_history_question)

Show the recent matches played by Scotland.


In [59]:
# Generate sample aggregation questions using real-style filters
sample_aggregation_questions = {
    "team_wins": generate_aggregation_question(
        intent="team_wins",
        team="Argentina",
        year=2022,
        tournament="FIFA World Cup"
    ),

    "team_goals_scored": generate_aggregation_question(
        intent="team_goals_scored",
        team="Argentina",
        year=2022,
        tournament="FIFA World Cup"
    ),

    "team_goals_conceded": generate_aggregation_question(
        intent="team_goals_conceded",
        team="Argentina",
        year=2022,
        tournament="FIFA World Cup"
    )
}

for intent, question in sample_aggregation_questions.items():
    print(f"{intent}:")
    print(question)
    print()

team_wins:
How many games did Argentina win during 2022?

team_goals_scored:
How many times did Argentina score in the FIFA World Cup?

team_goals_conceded:
What was Argentina's goals-conceded total in 2022?



In [61]:
# Test the improved aggregation-question generator
question, filter_type = generate_aggregation_question(
    intent="team_wins",
    team="Argentina",
    year=2022,
    tournament="FIFA World Cup"
)

print("Question:", question)
print("Filter type:", filter_type)

Question: How many wins did Argentina have in the FIFA World Cup?
Filter type: tournament


In [62]:
# Return all match-result records involving a selected team
def get_team_matches(dataframe, team):
    return dataframe[
        (dataframe["home_team"] == team) |
        (dataframe["away_team"] == team)
    ].copy()


# Apply either a year or tournament filter
def apply_match_filter(
    dataframe,
    filter_type,
    year=None,
    tournament=None
):
    if filter_type == "year":
        return dataframe[
            dataframe["date"].dt.year == int(year)
        ].copy()

    elif filter_type == "tournament":
        return dataframe[
            dataframe["tournament"] == tournament
        ].copy()

    elif filter_type == "none":
        return dataframe.copy()

    else:
        raise ValueError(
            f"Unsupported filter type: {filter_type}"
        )

In [63]:
# Retrieve all Argentina matches
argentina_matches = get_team_matches(
    results_clean,
    "Argentina"
)

print("All Argentina matches:", len(argentina_matches))

# Filter Argentina matches to the year 2022
argentina_2022_matches = apply_match_filter(
    argentina_matches,
    filter_type="year",
    year=2022
)

print("Argentina matches in 2022:", len(argentina_2022_matches))

# Display the first few filtered matches
display(
    argentina_2022_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament"
        ]
    ].head()
)

All Argentina matches: 1077
Argentina matches in 2022: 16


,date,home_team,away_team,home_score,away_score,tournament
44912,2022-01-27,Chile,Argentina,1,2,FIFA World Cup qualification
44942,2022-02-01,Argentina,Colombia,1,0,FIFA World Cup qualification
45010,2022-03-25,Argentina,Venezuela,3,0,FIFA World Cup qualification
45076,2022-03-29,Ecuador,Argentina,1,1,FIFA World Cup qualification
45151,2022-06-01,Italy,Argentina,0,3,CONMEBOL–UEFA Cup of Champions


In [64]:
# Generate answers for team-level aggregation intents
def generate_team_aggregation_answer(
    dataframe,
    team,
    intent,
    filter_type,
    year=None,
    tournament=None
):
    """
    Calculate an aggregated answer for a selected team.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        The cleaned match-results dataset.

    team : str
        The team whose statistics should be calculated.

    intent : str
        The aggregation intent to answer.

    filter_type : str
        The type of filtering used: year, tournament, or none.

    year : int or str, optional
        The year used when filter_type is year.

    tournament : str, optional
        The tournament used when filter_type is tournament.

    Returns
    -------
    str
        The calculated answer as text.
    """

    # Retrieve every match involving the selected team
    team_matches = get_team_matches(
        dataframe,
        team
    )

    # Apply the same filter used in the generated question
    filtered_matches = apply_match_filter(
        team_matches,
        filter_type=filter_type,
        year=year,
        tournament=tournament
    )

    # Return zero when no matching records exist
    if filtered_matches.empty:
        return "0"

    # Calculate the number of matches won by the selected team
    if intent == "team_wins":

        home_wins = (
            (filtered_matches["home_team"] == team) &
            (
                filtered_matches["home_score"] >
                filtered_matches["away_score"]
            )
        )

        away_wins = (
            (filtered_matches["away_team"] == team) &
            (
                filtered_matches["away_score"] >
                filtered_matches["home_score"]
            )
        )

        total_wins = (home_wins | away_wins).sum()

        return str(total_wins)

    # Calculate all goals scored by the selected team
    elif intent == "team_goals_scored":

        home_goals = filtered_matches.loc[
            filtered_matches["home_team"] == team,
            "home_score"
        ].sum()

        away_goals = filtered_matches.loc[
            filtered_matches["away_team"] == team,
            "away_score"
        ].sum()

        total_goals_scored = home_goals + away_goals

        return str(total_goals_scored)

    # Calculate all goals conceded by the selected team
    elif intent == "team_goals_conceded":

        goals_conceded_at_home = filtered_matches.loc[
            filtered_matches["home_team"] == team,
            "away_score"
        ].sum()

        goals_conceded_away = filtered_matches.loc[
            filtered_matches["away_team"] == team,
            "home_score"
        ].sum()

        total_goals_conceded = (
            goals_conceded_at_home +
            goals_conceded_away
        )

        return str(total_goals_conceded)

    else:
        raise ValueError(
            f"Unsupported team aggregation intent: {intent}"
        )

In [65]:
# Calculate Argentina's aggregated statistics for 2022
argentina_2022_wins = generate_team_aggregation_answer(
    dataframe=results_clean,
    team="Argentina",
    intent="team_wins",
    filter_type="year",
    year=2022
)

argentina_2022_goals_scored = generate_team_aggregation_answer(
    dataframe=results_clean,
    team="Argentina",
    intent="team_goals_scored",
    filter_type="year",
    year=2022
)

argentina_2022_goals_conceded = generate_team_aggregation_answer(
    dataframe=results_clean,
    team="Argentina",
    intent="team_goals_conceded",
    filter_type="year",
    year=2022
)

print("Argentina wins in 2022:", argentina_2022_wins)
print("Argentina goals scored in 2022:", argentina_2022_goals_scored)
print("Argentina goals conceded in 2022:", argentina_2022_goals_conceded)

Argentina wins in 2022: 12
Argentina goals scored in 2022: 41
Argentina goals conceded in 2022: 10


In [66]:
# Display Argentina's 2022 matches for manual validation
argentina_2022_validation = argentina_2022_matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament"
    ]
].sort_values("date")

display(argentina_2022_validation)

,date,home_team,away_team,home_score,away_score,tournament
44912,2022-01-27,Chile,Argentina,1,2,FIFA World Cup qualification
44942,2022-02-01,Argentina,Colombia,1,0,FIFA World Cup qualification
45010,2022-03-25,Argentina,Venezuela,3,0,FIFA World Cup qualification
45076,2022-03-29,Ecuador,Argentina,1,1,FIFA World Cup qualification
45151,2022-06-01,Italy,Argentina,0,3,CONMEBOL–UEFA Cup of Champions
45247,2022-06-05,Argentina,Estonia,5,0,Friendly
45503,2022-09-23,Argentina,Honduras,3,0,Friendly
45577,2022-09-27,Argentina,Jamaica,3,0,Friendly
45658,2022-11-16,United Arab Emirates,Argentina,0,5,Friendly
45721,2022-11-22,Argentina,Saudi Arabia,1,2,FIFA World Cup


In [67]:
# Intents that can be generated directly from one match-result row
direct_match_intents = [
    "match_score",
    "match_winner",
    "home_team_score",
    "away_team_score",
    "total_goals",
    "match_date",
    "match_location",
    "tournament",
    "neutral_status"
]

print("Direct match intents:", len(direct_match_intents))

Direct match intents: 9


In [68]:
# Number of generated training questions for each intent
QUESTIONS_PER_INTENT = 5000

print("Questions per intent:", QUESTIONS_PER_INTENT)
print(
    "Expected direct-match questions:",
    len(direct_match_intents) * QUESTIONS_PER_INTENT
)

Questions per intent: 5000
Expected direct-match questions: 45000


In [127]:
# Store all generated direct-match records
direct_match_records = []

# Generate an equal number of questions for every direct match intent
for intent in direct_match_intents:

    # Sample match records reproducibly
    sampled_matches = results_clean.sample(
        n=QUESTIONS_PER_INTENT,
        random_state=RANDOM_STATE
    )

    # Generate one labelled question-answer record per sampled match
    for _, row in sampled_matches.iterrows():

        record = create_match_qa_record(
            row=row,
            intent=intent
        )

        direct_match_records.append(record)

print(
    "Generated direct-match records:",
    len(direct_match_records)
)

Generated direct-match records: 45000


In [130]:
# Convert the generated records into a structured DataFrame
direct_match_qa_df = pd.DataFrame(
    direct_match_records
)

print("Shape:", direct_match_qa_df.shape)

display(direct_match_qa_df.head(10))

Shape: (45000, 4)


,question,intent,answer,source_dataset
0,What score did Philippines and Singapore finis...,match_score,Philippines 0 - 0 Singapore,results
1,How did the match between Kosovo and Madagasca...,match_score,Kosovo 1 - 0 Madagascar,results
2,What was the score between Palestine and Syria...,match_score,Palestine 1 - 1 Syria,results
3,What was the score between Tajikistan and Afgh...,match_score,Tajikistan 1 - 0 Afghanistan,results
4,How did the match between Netherlands and Indo...,match_score,Netherlands 9 - 2 Indonesia,results
5,What was the scoreline for Spain versus Uruguay?,match_score,Spain 2 - 1 Uruguay,results
6,What was the score between Denmark and Norway ...,match_score,Denmark 2 - 1 Norway,results
7,What was the result of Kenya against Sudan?,match_score,Kenya 2 - 0 Sudan,results
8,How did the match between England and Northern...,match_score,England 4 - 0 Northern Ireland,results
9,What was the score between New Zealand and Tur...,match_score,New Zealand 1 - 2 Turkey,results


In [73]:
# Count the generated questions for each direct match intent
direct_match_distribution = (
    direct_match_qa_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

display(direct_match_distribution)

,intent,question_count
0,match_score,5000
1,match_winner,5000
2,home_team_score,5000
3,away_team_score,5000
4,total_goals,5000
5,match_date,5000
6,match_location,5000
7,tournament,5000
8,neutral_status,5000


In [74]:
# Question templates for the scorer intent
question_templates["scorer"] = [
    "Who scored for {team} against {opponent}?",
    "Which player scored for {team} against {opponent}?",
    "Who was the goalscorer for {team} versus {opponent}?",
    "Who found the net for {team} against {opponent}?",
    "Which player got the goal for {team} against {opponent}?",
    "Who scored for {team} in the match against {opponent} on {date}?",
    "Who was on the scoresheet for {team} against {opponent}?",
    "Which player scored for {team} in the {date} match against {opponent}?"
]

In [75]:
# Question templates for the goal_minute intent
question_templates["goal_minute"] = [
    "In which minute did {scorer} score for {team} against {opponent}?",
    "When did {scorer} score against {opponent}?",
    "What minute was {scorer}'s goal for {team}?",
    "At what minute did {scorer} find the net against {opponent}?",
    "When was {scorer}'s goal scored in the match against {opponent}?",
    "What was the goal minute for {scorer} against {opponent} on {date}?",
    "In which minute did {scorer} score in {team} versus {opponent}?",
    "At what point in the match did {scorer} score against {opponent}?"
]

In [76]:
# Question templates for the penalty_status intent
question_templates["penalty_status"] = [
    "Was {scorer}'s goal for {team} against {opponent} a penalty?",
    "Did {scorer} score from the penalty spot against {opponent}?",
    "Was the goal by {scorer} against {opponent} scored from a penalty?",
    "Did {scorer}'s goal for {team} come from a penalty kick?",
    "Was {scorer}'s goal in {team} versus {opponent} a spot kick?",
    "Did {scorer} convert a penalty against {opponent} on {date}?",
    "Was the goal scored by {scorer} for {team} a penalty?",
    "Did {scorer} score a penalty in the match against {opponent}?"
]

In [77]:
# Question templates for the own_goal_status intent
question_templates["own_goal_status"] = [
    "Was {scorer}'s goal for {team} against {opponent} an own goal?",
    "Did {scorer} score an own goal against {opponent}?",
    "Was the goal by {scorer} recorded as an own goal?",
    "Did {scorer}'s goal in {team} versus {opponent} count as an own goal?",
    "Was {scorer}'s goal against {opponent} scored into their own net?",
    "Was the goal involving {scorer} on {date} an own goal?",
    "Did {scorer} register an own goal in the match against {opponent}?",
    "Was {scorer}'s goal for {team} marked as an own goal?"
]

In [78]:
# Verify the number of templates for each direct goalscorer intent
direct_goalscorer_intents = [
    "scorer",
    "goal_minute",
    "penalty_status",
    "own_goal_status"
]

for intent in direct_goalscorer_intents:
    print(f"{intent}: {len(question_templates[intent])} templates")

scorer: 8 templates
goal_minute: 8 templates
penalty_status: 8 templates
own_goal_status: 8 templates


In [79]:
# Generate one natural-language question from a goalscorer record
def generate_goalscorer_question(row, intent):
    """
    Generate one question for a direct goalscorer-related intent.

    Parameters
    ----------
    row : pandas.Series
        One row from the cleaned goalscorers dataset.

    intent : str
        The goalscorer intent used to select a template.

    Returns
    -------
    str
        A completed natural-language football question.
    """

    # Confirm that templates exist for the requested intent
    if intent not in question_templates:
        raise ValueError(
            f"No question templates found for intent: {intent}"
        )

    # Determine the opponent dynamically
    if row["team"] == row["home_team"]:
        opponent = row["away_team"]
    else:
        opponent = row["home_team"]

    # Select one template for the requested intent
    template = random.choice(
        question_templates[intent]
    )

    # Format the date consistently
    formatted_date = row["date"].strftime("%Y-%m-%d")

    # Replace placeholders using real goalscorer data
    question = template.format(
        team=row["team"],
        opponent=opponent,
        scorer=row["scorer"],
        date=formatted_date
    )

    return question

In [86]:
# Generate the expected answer for a direct goalscorer intent
def generate_goalscorer_answer(row, intent):
    """
    Generate the expected answer for one goalscorer record.
    """

    if intent == "scorer":
        return str(row["scorer"])

    elif intent == "goal_minute":

        # Return Unknown when the goal minute is missing
        if pd.isna(row["minute"]):
            return "Unknown"

        # Preserve the original football notation
        return str(row["minute"])

    elif intent == "penalty_status":
        return "Yes" if row["penalty"] else "No"

    elif intent == "own_goal_status":
        return "Yes" if row["own_goal"] else "No"

    else:
        raise ValueError(
            f"Unsupported goalscorer intent: {intent}"
        )

In [81]:
# Create one complete question-answer record from a goalscorer row
def create_goalscorer_qa_record(row, intent):
    """
    Create one labelled goalscorer question-answer record.
    """

    question = generate_goalscorer_question(
        row,
        intent
    )

    answer = generate_goalscorer_answer(
        row,
        intent
    )

    record = {
        "question": question,
        "intent": intent,
        "answer": answer,
        "source_dataset": "goalscorers"
    }

    return record

In [82]:
# Select one real goalscorer record for testing
sample_goal = scorers_clean.iloc[0]

# Generate one complete QA record for each goalscorer intent
sample_goal_records = []

for intent in direct_goalscorer_intents:
    record = create_goalscorer_qa_record(
        sample_goal,
        intent
    )

    sample_goal_records.append(record)

sample_goal_qa_df = pd.DataFrame(
    sample_goal_records
)

print("Goalscorer record:")
display(sample_goal.to_frame().T)

print("\nGenerated QA records:")
display(sample_goal_qa_df)

Goalscorer record:


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02 00:00:00,Chile,Uruguay,Uruguay,José Piendibene,44,False,False



Generated QA records:


,question,intent,answer,source_dataset
0,Who found the net for Uruguay against Chile?,scorer,José Piendibene,goalscorers
1,In which minute did José Piendibene score in U...,goal_minute,44,goalscorers
2,Was the goal by José Piendibene against Chile ...,penalty_status,No,goalscorers
3,Did José Piendibene score an own goal against ...,own_goal_status,No,goalscorers


In [88]:
# Store all generated direct goalscorer records
direct_goalscorer_records = []

# Generate an equal number of questions for every goalscorer intent
for intent in direct_goalscorer_intents:

    # Sample real goalscorer records reproducibly
    sampled_goals = scorers_clean.sample(
        n=QUESTIONS_PER_INTENT,
        random_state=RANDOM_STATE
    )

    # Create one labelled QA record from every sampled goal
    for _, row in sampled_goals.iterrows():

        record = create_goalscorer_qa_record(
            row=row,
            intent=intent
        )

        direct_goalscorer_records.append(record)

print(
    "Generated direct goalscorer records:",
    len(direct_goalscorer_records)
)

Generated direct goalscorer records: 20000


In [87]:
# Convert generated goalscorer records into a DataFrame
direct_goalscorer_qa_df = pd.DataFrame(
    direct_goalscorer_records
)

print("Shape:", direct_goalscorer_qa_df.shape)

display(direct_goalscorer_qa_df.head(10))

Shape: (5101, 4)


,question,intent,answer,source_dataset
0,Which player got the goal for Argentina agains...,scorer,Claudio Caniggia,goalscorers
1,Which player scored for Iran against Kyrgyzstan?,scorer,Farhad Majidi,goalscorers
2,Who scored for Spain in the match against Denm...,scorer,Andoni Goikoetxea Olaskoaga,goalscorers
3,Which player scored for Romania in the 1991-11...,scorer,Adrian Popescu,goalscorers
4,Who scored for Honduras in the match against U...,scorer,Julio César de León,goalscorers
5,Which player got the goal for Sierra Leone aga...,scorer,Lamin Conteh,goalscorers
6,Who was the goalscorer for Germany versus Bosn...,scorer,Tim Kleindienst,goalscorers
7,Which player scored for Paraguay in the 1965-0...,scorer,Juan Carlos Rojas,goalscorers
8,Who was on the scoresheet for United States ag...,scorer,Andrés Escobar,goalscorers
9,Who was on the scoresheet for Belgium against ...,scorer,Jan Ceulemans,goalscorers


In [89]:
# Count generated questions for each goalscorer intent
direct_goalscorer_distribution = (
    direct_goalscorer_qa_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

display(direct_goalscorer_distribution)

,intent,question_count
0,scorer,5000
1,goal_minute,101


In [90]:
# Count generated goal-minute questions with unknown answers
unknown_minute_count = direct_goalscorer_qa_df[
    (direct_goalscorer_qa_df["intent"] == "goal_minute") &
    (direct_goalscorer_qa_df["answer"] == "Unknown")
].shape[0]

print(
    "Goal-minute records with unknown answers:",
    unknown_minute_count
)

Goal-minute records with unknown answers: 0


In [91]:
# Inspect the datatype and unusual values in the minute column
print("Minute dtype:")
print(scorers_clean["minute"].dtype)

print()

print("Sample unique minute values:")
print(
    scorers_clean["minute"]
    .dropna()
    .astype(str)
    .sample(20, random_state=42)
    .tolist()
)

Minute dtype:
object

Sample unique minute values:
['50', '24', '35', '45', '49', '21', '15', '20', '60', '58', '87', '48', '35', '19', '10', '21', '9', '13', '80', '45']


In [92]:
# Find minute values that are not simple integers
non_integer_minutes = scorers_clean[
    ~scorers_clean["minute"]
    .astype(str)
    .str.fullmatch(r"\d+(\.0)?")
]

print("Non-standard minute records:")
print(len(non_integer_minutes))

display(
    non_integer_minutes[
        ["scorer", "minute"]
    ].head(20)
)

Non-standard minute records:
272


,scorer,minute
3347,Yiu Cheuk Yin,NaN
4059,Edward Acquah,NaN
4060,Edward Acquah,NaN
4063,Mengistu Worku,NaN
4064,Mengistu Worku,NaN
4065,Girma Tekle,NaN
4066,Girma Tesfaye,NaN
4067,Nasr Eddin Abbas,NaN
4068,Nasr Eddin Abbas,NaN
4069,Ibrahim Yahia El-Kawarty,NaN


In [93]:
# Recreate the DataFrame from the fully generated records list
direct_goalscorer_qa_df = pd.DataFrame(
    direct_goalscorer_records
)

print("Records stored in list:", len(direct_goalscorer_records))
print("DataFrame shape:", direct_goalscorer_qa_df.shape)

Records stored in list: 20000
DataFrame shape: (20000, 4)


In [94]:
# Recheck the class distribution
direct_goalscorer_distribution = (
    direct_goalscorer_qa_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

display(direct_goalscorer_distribution)

,intent,question_count
0,scorer,5000
1,goal_minute,5000
2,penalty_status,5000
3,own_goal_status,5000


In [95]:
# Count generated goal-minute questions with unknown answers
unknown_minute_count = direct_goalscorer_qa_df[
    (direct_goalscorer_qa_df["intent"] == "goal_minute") &
    (direct_goalscorer_qa_df["answer"] == "Unknown")
].shape[0]

print(
    "Goal-minute records with unknown answers:",
    unknown_minute_count
)


Goal-minute records with unknown answers: 28


In [96]:
# Question templates for the shootout_winner intent
question_templates["shootout_winner"] = [
    "Who won the penalty shootout between {home_team} and {away_team}?",
    "Which team won the shootout between {home_team} and {away_team}?",
    "Who was the winner of the penalty shootout between {home_team} and {away_team}?",
    "Who won the shootout when {home_team} played {away_team}?",
    "Which side won the penalty shootout between {home_team} and {away_team}?",
    "Who won the penalty shootout between {home_team} and {away_team} on {date}?",
    "Who won the shootout involving {home_team} and {away_team}?",
    "Which team was victorious in the penalty shootout between {home_team} and {away_team}?"
]

In [97]:
# Question templates for the first_shooter intent
question_templates["first_shooter"] = [
    "Which team took the first penalty between {home_team} and {away_team}?",
    "Who took the first penalty in the shootout between {home_team} and {away_team}?",
    "Which team kicked first in the shootout between {home_team} and {away_team}?",
    "Who was the first shooter in the penalty shootout between {home_team} and {away_team}?",
    "Which side took the opening penalty between {home_team} and {away_team}?",
    "Who started the penalty shootout between {home_team} and {away_team}?",
    "Which team took the first kick between {home_team} and {away_team} on {date}?",
    "Who kicked first when {home_team} played {away_team} in a penalty shootout?"
]

In [98]:
direct_shootout_intents = [
    "shootout_winner",
    "first_shooter"
]

for intent in direct_shootout_intents:
    print(f"{intent}: {len(question_templates[intent])} templates")

shootout_winner: 8 templates
first_shooter: 8 templates


In [99]:
# Generate one question from a penalty-shootout record
def generate_shootout_question(row, intent):

    if intent not in question_templates:
        raise ValueError(
            f"No question templates found for intent: {intent}"
        )

    template = random.choice(question_templates[intent])

    formatted_date = row["date"].strftime("%Y-%m-%d")

    return template.format(
        home_team=row["home_team"],
        away_team=row["away_team"],
        date=formatted_date
    )


# Generate the correct answer for a shootout question
def generate_shootout_answer(row, intent):

    if intent == "shootout_winner":
        return str(row["winner"])

    elif intent == "first_shooter":
        return str(row["first_shooter"])

    else:
        raise ValueError(
            f"Unsupported shootout intent: {intent}"
        )


# Create one complete shootout QA record
def create_shootout_qa_record(row, intent):

    return {
        "question": generate_shootout_question(row, intent),
        "intent": intent,
        "answer": generate_shootout_answer(row, intent),
        "source_dataset": "shootouts"
    }

In [100]:
# Generate shootout questions using every record and every template
direct_shootout_records = []

for intent in direct_shootout_intents:

    for _, row in shootouts_clean.iterrows():

        for template in question_templates[intent]:

            formatted_date = row["date"].strftime("%Y-%m-%d")

            question = template.format(
                home_team=row["home_team"],
                away_team=row["away_team"],
                date=formatted_date
            )

            record = {
                "question": question,
                "intent": intent,
                "answer": generate_shootout_answer(row, intent),
                "source_dataset": "shootouts"
            }

            direct_shootout_records.append(record)

direct_shootout_qa_df = pd.DataFrame(
    direct_shootout_records
)

print("Shootout dataset shape:", direct_shootout_qa_df.shape)

display(
    direct_shootout_qa_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

Shootout dataset shape: (10912, 4)


,intent,question_count
0,shootout_winner,5456
1,first_shooter,5456


In [129]:
# Combine all generated QA datasets
question_intent_dataset = pd.concat(
    [
        direct_match_qa_df,
        direct_goalscorer_qa_df,
        direct_shootout_qa_df
    ],
    ignore_index=True
)

print("Combined dataset shape:")
print(question_intent_dataset.shape)

display(question_intent_dataset.head())

Combined dataset shape:
(75912, 4)


,question,intent,answer,source_dataset
0,What score did Philippines and Singapore finis...,match_score,Philippines 0 - 0 Singapore,results
1,How did the match between Kosovo and Madagasca...,match_score,Kosovo 1 - 0 Madagascar,results
2,What was the score between Palestine and Syria...,match_score,Palestine 1 - 1 Syria,results
3,What was the score between Tajikistan and Afgh...,match_score,Tajikistan 1 - 0 Afghanistan,results
4,How did the match between Netherlands and Indo...,match_score,Netherlands 9 - 2 Indonesia,results


In [131]:
# Shuffle the dataset
question_intent_dataset = (
    question_intent_dataset
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

print("Dataset shuffled successfully.")

Dataset shuffled successfully.


In [103]:
# Count duplicate questions
duplicate_questions = (
    question_intent_dataset["question"]
    .duplicated()
    .sum()
)

print("Duplicate questions:", duplicate_questions)

Duplicate questions: 5234


In [104]:
# Distribution of all intents
intent_distribution = (
    question_intent_dataset["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

display(intent_distribution)

,intent,question_count
0,shootout_winner,5456
1,first_shooter,5456
2,home_team_score,5000
3,total_goals,5000
4,goal_minute,5000
5,match_score,5000
6,scorer,5000
7,away_team_score,5000
8,match_location,5000
9,neutral_status,5000


In [106]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
/content/football-qa-nlp


In [107]:
import os

print(os.path.exists("models/features"))

True


In [109]:
print(question_intent_dataset.shape)

print()

print(question_intent_dataset["intent"].value_counts())

print()

duplicate_questions = (
    question_intent_dataset["question"]
    .duplicated()
    .sum()
)

print("Duplicate questions:", duplicate_questions)

(75912, 4)

intent
shootout_winner    5456
first_shooter      5456
home_team_score    5000
total_goals        5000
goal_minute        5000
match_score        5000
scorer             5000
away_team_score    5000
match_location     5000
neutral_status     5000
match_winner       5000
penalty_status     5000
own_goal_status    5000
tournament         5000
match_date         5000
Name: count, dtype: int64

Duplicate questions: 5234


In [108]:
question_intent_dataset.to_csv(
    "models/features/question_intent_dataset.csv",
    index=False
)

print("Complete QA dataset saved successfully.")

Complete QA dataset saved successfully.


In [110]:
# Check whether the same question appears under multiple intent labels
question_intent_counts = (
    question_intent_dataset
    .groupby("question")["intent"]
    .nunique()
)

conflicting_questions = question_intent_counts[
    question_intent_counts > 1
]

print(
    "Questions assigned to multiple intents:",
    len(conflicting_questions)
)

Questions assigned to multiple intents: 196


In [111]:
# Display questions that were assigned to more than one intent
conflicting_examples = (
    question_intent_dataset[
        question_intent_dataset["question"].isin(
            conflicting_questions.index
        )
    ]
    .sort_values("question")
)

display(
    conflicting_examples[
        ["question", "intent", "answer", "source_dataset"]
    ].head(40)
)

,question,intent,answer,source_dataset
18980,How many goals did Aruba score against Curaçao?,away_team_score,0,results
37661,How many goals did Aruba score against Curaçao?,home_team_score,1,results
27076,How many goals did Austria score against Bosni...,home_team_score,1,results
15076,How many goals did Austria score against Bosni...,home_team_score,1,results
41032,How many goals did Austria score against Bosni...,away_team_score,2,results
54710,How many goals did Austria score in the Friend...,away_team_score,1,results
18655,How many goals did Austria score in the Friend...,away_team_score,1,results
29400,How many goals did Austria score in the Friend...,home_team_score,1,results
58200,How many goals did Barbados score against Guyana?,home_team_score,0,results
4821,How many goals did Barbados score against Guyana?,away_team_score,2,results


In [112]:
# Show the intent combinations causing conflicts
conflict_summary = (
    conflicting_examples
    .groupby("question")["intent"]
    .apply(lambda values: tuple(sorted(set(values))))
    .value_counts()
    .reset_index()
)

conflict_summary.columns = [
    "conflicting_intents",
    "number_of_questions"
]

display(conflict_summary)

,conflicting_intents,number_of_questions
0,"(away_team_score, home_team_score)",196


In [123]:
question_intent_counts = (
    question_intent_dataset
    .groupby("question")["intent"]
    .nunique()
)

conflicting_questions = question_intent_counts[
    question_intent_counts > 1
]

print(len(conflicting_questions))

167


In [124]:
conflicting_examples = (
    question_intent_dataset[
        question_intent_dataset["question"].isin(conflicting_questions.index)
    ]
    .sort_values("question")
)

display(
    conflicting_examples[
        ["question", "intent"]
    ].head(50)
)

,question,intent
75411,How many goals did Brazil score in the Friendl...,away_team_score
23913,How many goals did Brazil score in the Friendl...,home_team_score
20065,How many goals did Bulgaria score in the Frien...,away_team_score
50582,How many goals did Bulgaria score in the Frien...,home_team_score
44785,How many goals did Congo score in the African ...,home_team_score
55344,How many goals did Congo score in the African ...,away_team_score
55422,How many goals did France score in the Friendl...,home_team_score
60531,How many goals did France score in the Friendl...,away_team_score
34109,How many goals did Germany score in the Friend...,home_team_score
9963,How many goals did Germany score in the Friend...,away_team_score


In [132]:
question_intent_counts = (
    question_intent_dataset
    .groupby("question")["intent"]
    .nunique()
)

conflicting_questions = question_intent_counts[
    question_intent_counts > 1
]

print(
    "Questions assigned to multiple intents:",
    len(conflicting_questions)
)

Questions assigned to multiple intents: 0


In [133]:
# Save the complete question-intent dataset
question_intent_dataset.to_csv(
    os.path.join(
        features_path,
        "question_intent_dataset.csv"
    ),
    index=False
)

print("Question-intent dataset saved successfully.")

Question-intent dataset saved successfully.


In [134]:
from sklearn.model_selection import train_test_split

In [135]:
# Split into train (80%) and temporary (20%)
train_df, temp_df = train_test_split(
    question_intent_dataset,
    test_size=0.20,
    stratify=question_intent_dataset["intent"],
    random_state=RANDOM_STATE
)

# Split temporary into validation (10%) and test (10%)
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["intent"],
    random_state=RANDOM_STATE
)

In [136]:
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Testing:", test_df.shape)

Training: (60729, 4)
Validation: (7591, 4)
Testing: (7592, 4)


In [137]:
train_df.to_csv(
    os.path.join(features_path, "train_dataset.csv"),
    index=False
)

validation_df.to_csv(
    os.path.join(features_path, "validation_dataset.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(features_path, "test_dataset.csv"),
    index=False
)

print("Train, validation and test datasets saved.")

Train, validation and test datasets saved.


In [138]:
print("Train")
print(train_df["intent"].value_counts())

print("\nValidation")
print(validation_df["intent"].value_counts())

print("\nTest")
print(test_df["intent"].value_counts())

Train
intent
first_shooter      4365
shootout_winner    4364
match_date         4000
total_goals        4000
penalty_status     4000
match_winner       4000
goal_minute        4000
tournament         4000
match_score        4000
scorer             4000
match_location     4000
neutral_status     4000
home_team_score    4000
own_goal_status    4000
away_team_score    4000
Name: count, dtype: int64

Validation
intent
shootout_winner    546
first_shooter      545
penalty_status     500
total_goals        500
goal_minute        500
match_location     500
home_team_score    500
scorer             500
match_date         500
own_goal_status    500
neutral_status     500
tournament         500
match_winner       500
match_score        500
away_team_score    500
Name: count, dtype: int64

Test
intent
shootout_winner    546
first_shooter      546
total_goals        500
match_date         500
goal_minute        500
own_goal_status    500
away_team_score    500
penalty_status     500
match_location

In [139]:
print("Saved feature files:\n")

for filename in sorted(os.listdir(features_path)):
    print(filename)

Saved feature files:

football_corpus_clean.csv
lstm_padded_sequences.npy
lstm_tokenizer.pkl
max_sequence_length.pkl
question_intent_dataset.csv
test_dataset.csv
tfidf_features.npz
tfidf_vectorizer.pkl
train_dataset.csv
validation_dataset.csv


In [140]:
print(features_path)

print("\nFiles:")
for f in sorted(os.listdir(features_path)):
    print(f)

/content/football-qa-nlp/models/features

Files:
football_corpus_clean.csv
lstm_padded_sequences.npy
lstm_tokenizer.pkl
max_sequence_length.pkl
question_intent_dataset.csv
test_dataset.csv
tfidf_features.npz
tfidf_vectorizer.pkl
train_dataset.csv
validation_dataset.csv
